# Tutorial: TF knock-in perturbation inference

This tutorial demonstrates the full workflow for TF knock-in perturbation inference:

1. Load input resources and define paths
2. Read one TF-up DEG list and one TF-down DEG list
3. Build wild-type inputs for the selected genes
4. Scan motifs and generate knock-in inputs with a fixed donor peak signal
5. Convert text inputs into model-ready tensors
6. Run the pretrained model on WT and KI inputs
7. Evaluate predicted delta direction and correlation with log2FC

This tutorial is intentionally lightweight:
- it only uses **two CSV files** for the same TF: one `up` and one `down`
- it supports optional subsampling for limited compute
- it reports only the main metrics:
  - overall accuracy
  - up accuracy
  - down accuracy
  - macro-F1
  - Pearson correlation between predicted delta and log2FC

In [1]:
import os
import sys
import json
import random
import numpy as np
import pandas as pd
import torch

from models import *
from data_processing import *

from toolkit import (
    GeneATACPreprocessor,
    FIMOMotifScanner,
    TFKIPeakInserterFixedSignal,
    load_fixed_donor_signal,
    prepare_tutorial_gene_rows,
    collect_gene_names_from_rows,
    build_tutorial_directory_tree,
    build_wild_type_inputs_for_genes,
    build_wt_dataset_for_tutorial,
    build_ki_dataset_for_tutorial,
    load_model_components,
    predict_delta_for_common_genes,
    summarize_results,
    print_summary,
    save_results_table,
    save_summary_json,
    save_gene_rows_as_csv
)

In [ ]:
# =========================
# Required input resources
# =========================

# Reference annotation and genome
GTF_PATH = "../example_data/genes-2020-A.gtf"   ## gene annotation file, can be downloaded from 10x Genomics
GENOME_DIR = "../example_data/genome"           ## GRCh38 assembly, sequence of each chromatin was saved as .pkl file

# WT ATAC resources used to extract wild-type peaks/signals
PEAK_PATH = "../example_data/ENCFF333TAT.narrowPeak"    ## ATAC peak file, can be downloaded from ENCODE
BIGWIG_PATH = "../example_data/ENCFF357GNC.bigWig"      ## ATAC signal file, can be downloaded from ENCODE

# Motif directory and FIMO executable
MOTIF_DIR = "../example_data/motifs_ki"
FIMO_PATH = "fimo"
FIXED_DONOR_SIGNAL_PATH = "../example_data/fixed_donor_signal.pt"

# Pretrained model checkpoint directory
CHECKPOINT_DIR = "../checkpoints"    ## path of pretrained model parameters 

# k-mer dictionary required by your original preprocessing pipeline
GENOME_DICT_PATH = "../example_data/kmer_dict.pkl"

# Two DEG CSV files for the same TF:
# one up-regulated gene list and one down-regulated gene list
UP_CSV_PATH = "../example_data/K562_CEBPA_up.csv"        ## DEGs in those two files were computed by analyzing the perturb-seq results.
DOWN_CSV_PATH = "../example_data/K562_CEBPA_down.csv"

# Output root for this tutorial run
TUTORIAL_ROOT = "../tf_kd_output"        ## define your output path

# =========================
# Runtime options
# =========================

CUDA_DEVICE = 0
RANDOM_SEED = 34

# Optional gene subsampling:
# None -> use all genes in each CSV
# integer -> sample up to this many genes from each direction
MAX_SAMPLES_PER_DIRECTION = 50

# Optional evaluation subsampling after successful preprocessing
# None -> evaluate all common genes
# integer -> randomly evaluate only part of the final WT/KI overlap
MAX_EVAL_SAMPLES = None

# Motif threshold
MOTIF_PVALUE_THRESH = 1e-4

# Sequence/window settings
SEQ_LEN = 1029
WINDOW = 150000

# Name for this tutorial run
RUN_NAME = "tf_ki_tutorial_run"


In [3]:
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)

paths = build_tutorial_directory_tree(
    tutorial_root=TUTORIAL_ROOT,
    run_name=RUN_NAME
)

print("Output directories:")
for k, v in paths.items():
    print(f"  {k}: {v}")


Output directories:
  run_dir: /mnt/nas-data/yzk/pertformer-tutorials/tf_kd_output/tf_ki_tutorial_run
  wt_dir: /mnt/nas-data/yzk/pertformer-tutorials/tf_kd_output/tf_ki_tutorial_run/wild_type_inputs
  ki_dir: /mnt/nas-data/yzk/pertformer-tutorials/tf_kd_output/tf_ki_tutorial_run/ki_inputs
  meta_dir: /mnt/nas-data/yzk/pertformer-tutorials/tf_kd_output/tf_ki_tutorial_run/meta
  result_dir: /mnt/nas-data/yzk/pertformer-tutorials/tf_kd_output/tf_ki_tutorial_run/results


In [4]:
gene_rows, cell_line, tf_name, run_prefix = prepare_tutorial_gene_rows(
    up_csv_path=UP_CSV_PATH,
    down_csv_path=DOWN_CSV_PATH,
    max_samples_per_direction=MAX_SAMPLES_PER_DIRECTION,
    random_seed=RANDOM_SEED
)

gene_names = collect_gene_names_from_rows(gene_rows)

print(f"Selected genes for tutorial: {len(gene_names)}")
print(f"Cell line: {cell_line}")
print(f"TF: {tf_name}")

tutorial_gene_table_path = os.path.join(paths["meta_dir"], f"{run_prefix}_selected_gene_rows.csv")
save_gene_rows_as_csv(gene_rows, tutorial_gene_table_path)


[Tutorial] Merged up/down gene rows: 100
[Tutorial] Cell line: K562
[Tutorial] TF: CEBPA
Selected genes for tutorial: 100
Cell line: K562
TF: CEBPA
[Output] Tutorial gene table saved to: /mnt/nas-data/yzk/pertformer-tutorials/tf_kd_output/tf_ki_tutorial_run/meta/K562_CEBPA_selected_gene_rows.csv


In [5]:
print("[Step] Initializing preprocessing resources...")

preprocessor = GeneATACPreprocessor(
    gtf_path=GTF_PATH,
    genome_dir=GENOME_DIR,
    peak_path=PEAK_PATH,
    bigwig_path=BIGWIG_PATH
)

motif_scanner = FIMOMotifScanner(
    motif_dir=MOTIF_DIR,
    fimo_path=FIMO_PATH
)

fixed_donor_signal = load_fixed_donor_signal(FIXED_DONOR_SIGNAL_PATH)

genome_dict = torch.load(GENOME_DICT_PATH)

print("[Step] Resource initialization complete.")


[Step] Initializing preprocessing resources...
[Preprocessor] Loading static resources...
[Preprocessor] GTF loaded: 36591 genes


/home/yangzikun.yzk/tutorials/Tutorials_TF_overexpression/toolkit.py:103: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  self.genome[i] = torch.load(f"{genome_dir}/chr{i+1}.p

[Preprocessor] Genome tensors loaded
[Preprocessor] Peaks loaded: 161693 unique peaks
[Preprocessor] BigWig opened
[Preprocessor] Ready.

[Donor] Loaded fixed donor signal: length=1029
[Step] Resource initialization complete.


/home/yangzikun.yzk/tutorials/Tutorials_TF_overexpression/toolkit.py:309: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  obj = torch.load(saved_path, map_location="cpu")
/tmp

In [6]:
print("[Step] Building wild-type text inputs...")

wt_meta_path = build_wild_type_inputs_for_genes(
    preprocessor=preprocessor,
    gene_names=gene_names,
    wild_type_dir=paths["wt_dir"]
)

print(f"WT meta saved at: {wt_meta_path}")


[Step] Building wild-type text inputs...
[WT] Building WT inputs for 100 genes...
[WT] Progress: 1/100 | success=0
[WT] Progress: 10/100 | success=9
[WT] Progress: 20/100 | success=19
[WT] Progress: 30/100 | success=29
[WT] Progress: 40/100 | success=38
[WT] Progress: 50/100 | success=45
[WT] Progress: 60/100 | success=49
[WT] Progress: 70/100 | success=55
[WT] Progress: 80/100 | success=61
[WT] Progress: 90/100 | success=68
[WT] Progress: 100/100 | success=78
[WT] Done. Success=79/100
WT meta saved at: /mnt/nas-data/yzk/pertformer-tutorials/tf_kd_output/tf_ki_tutorial_run/wild_type_inputs/meta_info.json


In [7]:
print("[Step] Building knock-in text inputs...")

ki_meta_path = os.path.join(paths["meta_dir"], f"{run_prefix}_ki_meta.json")

ki_builder = TFKIPeakInserterFixedSignal(
    preprocessor=preprocessor,
    wild_type_dir=paths["wt_dir"],
    ki_dir=paths["ki_dir"],
    motif_scanner=motif_scanner,
    fixed_donor_signal=fixed_donor_signal,
    motif_pvalue_thresh=MOTIF_PVALUE_THRESH,
    window=WINDOW,
    seq_len=SEQ_LEN
)

ki_meta_records = ki_builder.batch_build_for_gene_rows(
    tf_name=tf_name,
    gene_rows=gene_rows,
    meta_output_path=ki_meta_path
)

print(f"KI meta saved at: {ki_meta_path}")


[Step] Building knock-in text inputs...
[KI] Building KI inputs for TF=CEBPA on 100 genes...
[KI] Progress: 1/100 | success=0
[KI] Progress: 20/100 | success=19
[KI] Progress: 30/100 | success=29
[KI] Progress: 50/100 | success=45
[KI] Progress: 60/100 | success=49
[KI] Progress: 70/100 | success=55
[KI] Progress: 80/100 | success=61
[KI] Progress: 90/100 | success=68
[KI] Progress: 100/100 | success=78
[KI] Done. Success=79/100
[KI] Meta saved to: /mnt/nas-data/yzk/pertformer-tutorials/tf_kd_output/tf_ki_tutorial_run/meta/K562_CEBPA_ki_meta.json
KI meta saved at: /mnt/nas-data/yzk/pertformer-tutorials/tf_kd_output/tf_ki_tutorial_run/meta/K562_CEBPA_ki_meta.json


In [8]:
print("[Step] Converting WT inputs into model-ready tensors...")

wt_dataset, gene_to_wt_sample = build_wt_dataset_for_tutorial(
    wild_type_dir=paths["wt_dir"],
    wt_meta_path=wt_meta_path,
    gene_names=gene_names,
    genome_dict=genome_dict,
    narrowPeak_Reader=narrowPeak_Reader,
    pre_processing=pre_processing,
    tokenizer=tokenizer,
    pre_pro=pre_pro
)

print(f"WT dataset size: {len(wt_dataset)}")


[Step] Converting WT inputs into model-ready tensors...
[Dataset] Building WT dataset for 100 genes...
[Dataset][WT] Progress: 1/100 | kept=0
[Dataset][WT] Progress: 10/100 | kept=9
[Dataset][WT] Progress: 20/100 | kept=19
[Dataset][WT] Progress: 30/100 | kept=29
[Dataset][WT] Progress: 40/100 | kept=38
[Dataset][WT] Progress: 50/100 | kept=45
[Dataset][WT] Progress: 60/100 | kept=49
[Dataset][WT] Progress: 70/100 | kept=55
[Dataset][WT] Progress: 80/100 | kept=61
[Dataset][WT] Progress: 90/100 | kept=68
[Dataset][WT] Progress: 100/100 | kept=78
[Dataset][WT] Done. Samples=79
WT dataset size: 79


In [9]:
print("[Step] Converting KI inputs into model-ready tensors...")

ki_dataset = build_ki_dataset_for_tutorial(
    ki_dir=paths["ki_dir"],
    ki_meta_path=ki_meta_path,
    gene_to_wt_sample=gene_to_wt_sample,
    genome_dict=genome_dict,
    narrowPeak_Reader=narrowPeak_Reader,
    pre_processing=pre_processing,
    tokenizer=tokenizer,
    pre_pro=pre_pro
)

print(f"KI dataset size: {len(ki_dataset)}")


[Step] Converting KI inputs into model-ready tensors...
[Dataset] Building KI dataset...
[Dataset][KI] Progress: 1/238 | kept=0
[Dataset][KI] Progress: 10/238 | kept=1
[Dataset][KI] Progress: 20/238 | kept=7
[Dataset][KI] Progress: 30/238 | kept=12
[Dataset][KI] Progress: 40/238 | kept=15
[Dataset][KI] Progress: 50/238 | kept=18
[Dataset][KI] Progress: 60/238 | kept=20
[Dataset][KI] Progress: 70/238 | kept=24
[Dataset][KI] Progress: 80/238 | kept=24
[Dataset][KI] Progress: 90/238 | kept=26
[Dataset][KI] Progress: 100/238 | kept=27
[Dataset][KI] Progress: 110/238 | kept=31
[Dataset][KI] Progress: 120/238 | kept=36
[Dataset][KI] Progress: 130/238 | kept=38
[Dataset][KI] Progress: 140/238 | kept=39
[Dataset][KI] Progress: 150/238 | kept=39
[Dataset][KI] Progress: 160/238 | kept=44
[Dataset][KI] Progress: 170/238 | kept=47
[Dataset][KI] Progress: 180/238 | kept=50
[Dataset][KI] Progress: 190/238 | kept=52
[Dataset][KI] Progress: 200/238 | kept=56
[Dataset][KI] Progress: 210/238 | kept=59
[

In [10]:
print("[Step] Loading pretrained model...")

model_parts = load_model_components(
    checkpoint_dir=CHECKPOINT_DIR,
    cuda_device=CUDA_DEVICE
)


[Step] Loading pretrained model...
[Model] Loading model components...
[Model] Model loaded.



In [11]:
print("[Step] Running inference...")

result_df = predict_delta_for_common_genes(
    wt_dataset=wt_dataset,
    ki_dataset=ki_dataset,
    model_parts=model_parts,
    max_eval_samples=MAX_EVAL_SAMPLES,
    random_seed=RANDOM_SEED
)

print(f"Prediction table shape: {result_df.shape}")
result_df.head()


[Step] Running inference...
[Inference] Common genes between WT and KI: 68
[Inference] Using all 68 genes
[Inference] Progress: 1/68
[Inference] Progress: 10/68
[Inference] Progress: 20/68
[Inference] Progress: 30/68
[Inference] Progress: 40/68
[Inference] Progress: 50/68
[Inference] Progress: 60/68
[Inference] Progress: 68/68
[Inference] Done. Successful predictions: 68
Prediction table shape: (68, 5)


,gene_name,pred_delta,pred_direction,true_direction,log2FC
0,ADGRE2,4.593750,up,up,4.654994
1,AK8,1.191406,up,down,-1.100433
2,ANXA1,5.468750,up,up,7.431941
3,ARAP3,1.640625,up,up,5.725296
4,ARL6IP5,4.109375,up,up,0.921933


In [12]:
print("[Step] Computing summary metrics...")

summary = summarize_results(result_df)
print_summary(summary)

[Step] Computing summary metrics...

================ Tutorial Summary ================
Evaluated samples      : 68
Accuracy               : 0.7647
Macro-F1               : 0.7543
Up accuracy            : 0.8462
Down accuracy          : 0.6552

